# 🎙️ VoiceBatch Studio v0.0 [Super-Fast Edition]
अपलोड और जनरेशन की स्पीड को फिक्स कर दिया गया है।

In [ ]:
# @title 🛠️ Step 1: सेटअप (बिना किसी एरर के)
import os
from google.colab import drive
print("⏳ जरूरी फाइल्स लोड हो रही हैं...")
!pip install -q coqpit-config coqui-tts gradio librosa soundfile
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
os.makedirs("outputs", exist_ok=True)
print("✅ ड्राइव कनेक्ट हो गई!")

In [ ]:
# @title 🚀 Step 2: हाई-स्पीड जनरेटर चालू करें
import gradio as gr
import torch, librosa, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/"

print("⏳ इंजन गरम हो रहा है (Turbo Mode)...")
tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)

def fast_engine(text, audio_sample, silence_rem):
    if not audio_sample or not text: return None
    
    # जनरेशन स्पीड बढ़ाने के लिए इन्फरेंस मोड
    with torch.inference_mode():
        parts = re.split(r'(?<=[।?!])\s+', text)
        combined = []
        for p in parts:
            if len(p.strip()) < 2: continue
            wav = tts.tts(text=p, speaker_wav=audio_sample, language='hi')
            combined.append(np.array(wav))
        
        final = np.concatenate(combined)
        if silence_rem: final, _ = librosa.effects.trim(final, top_db=20)
        
        out_path = "outputs/v0_result.wav"
        sf.write(out_path, final, 24000)
        return out_path

with gr.Blocks(theme=gr.themes.Default()) as demo:
    gr.Markdown("# 🎙️ VoiceBatch Studio v0.0")
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label="Script (Hindi)", lines=8)
            cnt = gr.Markdown("Shabd: 0")
            txt.change(lambda x: f"Shabd: {len(x.split())}", inputs=[txt], outputs=[cnt])
            
            # अपलोड स्पीड के लिए बदलाव
            smp = gr.Audio(label="Upload Sample", type='filepath')
            
            sil = gr.Checkbox(label="Silence Remover", value=True)
            btn = gr.Button("Audio Banayein ⚡", variant="primary")
        with gr.Column():
            res = gr.Audio(label="Result")

    btn.click(fast_engine, [txt, smp, sil], res)

# 'share=True' के साथ 'concurrency_count' को बढ़ाया गया है
demo.queue().launch(share=True, debug=True)